Notebook 01 - Auditoria da Base Maicon

Este notebook realiza a auditoria da base documental fornecida pelo professor, verificando a estrutura dos arquivos, identificando os artigos disponíveis, validando os arquivos JSON de chunks e gerando uma tabela mestre da base.

O objetivo é garantir que a base esteja consistente antes da geração dos embeddings e da construção do índice FAISS.

Preparação do Ambiente

In [ ]:
# Importa as bibliotecas necessárias

import json
import pandas as pd

from pathlib import Path

from google.colab import drive

In [ ]:
# Monta o Google Drive para acessar a base documental

drive.mount(
    "/content/drive"
)

Mounted at /content/drive


Funções Auxiliares

In [ ]:
# Define funções auxiliares para padronizar as mensagens

def print_header(title):
    print("\n" + "=" * 70)
    print(f" {title}")
    print("=" * 70)


def print_success(message):
    print(f"\n✅ {message}")


def print_error(message):
    print(f"\n❌ {message}")


def print_warning(message):
    print(f"\n⚠️ {message}")


def print_info(label, value):
    print(f"{label:<25} {value}")

Configuração do Projeto

In [ ]:
# Define os caminhos utilizados no projeto

PROJECT_PATH = Path(
    "/content/drive/MyDrive/RAG_Novo_Embeddings"
)

BASE_DIR = (
    PROJECT_PATH
    / "01_Base_Maicon"
)

EMBEDDINGS_DIR = (
    PROJECT_PATH
    / "02_Embeddings"
)

FAISS_DIR = (
    PROJECT_PATH
    / "03_FAISS"
)

RESULTS_DIR = (
    PROJECT_PATH
    / "04_Resultados"
)

NOTEBOOKS_DIR = (
    PROJECT_PATH
    / "05_Notebooks_RAG"
)

MANIFEST_FILE = (
    RESULTS_DIR
    / "manifesto_base.csv"
)

print_header(
    "CONFIGURAÇÃO DO PROJETO"
)

print_info(
    "Projeto:",
    PROJECT_PATH
)

print_info(
    "Base:",
    BASE_DIR
)

print_info(
    "Manifesto:",
    MANIFEST_FILE
)

print_success(
    "Caminhos configurados com sucesso."
)

print("=" * 70)


 CONFIGURAÇÃO DO PROJETO
Projeto:                  /content/drive/MyDrive/RAG_Novo_Embeddings
Base:                     /content/drive/MyDrive/RAG_Novo_Embeddings/01_Base_Maicon
Manifesto:                /content/drive/MyDrive/RAG_Novo_Embeddings/04_Resultados/manifesto_base.csv

✅ Caminhos configurados com sucesso.


Verificação da Base

In [ ]:
# Verifica se a pasta da base existe

print_header(
    "VERIFICAÇÃO DA BASE"
)

if not BASE_DIR.exists():

    raise FileNotFoundError(
        f"Pasta não encontrada:\n{BASE_DIR}"
    )

print_success(
    "Pasta da base localizada com sucesso."
)

print("=" * 70)


 VERIFICAÇÃO DA BASE

✅ Pasta da base localizada com sucesso.


In [ ]:
# Localiza os arquivos presentes na base

print_header(
    "LOCALIZAÇÃO DOS ARQUIVOS"
)

json_files = sorted(
    BASE_DIR.glob(
        "*_chunksrecord_*.json"
    )
)

txt_files = sorted(
    BASE_DIR.glob(
        "*.txt"
    )
)

csv_files = sorted(
    BASE_DIR.glob(
        "*.csv"
    )
)

print_info(
    "Arquivos JSON:",
    len(json_files)
)

print_info(
    "Arquivos TXT:",
    len(txt_files)
)

print_info(
    "Arquivos CSV:",
    len(csv_files)
)

print_info(
    "Total de arquivos:",
    len(json_files)
    + len(txt_files)
    + len(csv_files)
)

if not json_files:

    raise FileNotFoundError(
        "Nenhum arquivo chunksrecord JSON foi encontrado."
    )

print_success(
    "Arquivos localizados com sucesso."
)

print("=" * 70)


 LOCALIZAÇÃO DOS ARQUIVOS
Arquivos JSON:            648
Arquivos TXT:             701
Arquivos CSV:             1
Total de arquivos:        1350

✅ Arquivos localizados com sucesso.


Auditoria dos Arquivos JSON

In [ ]:
# Valida cada arquivo JSON e prepara os dados da tabela mestre

print_header(
    "AUDITORIA DOS ARQUIVOS JSON"
)

manifest_records = []

invalid_files = []

for file in json_files:

    try:

        data = json.loads(
            file.read_text(
                encoding="utf-8"
            )
        )

        if not isinstance(
            data,
            list
        ):

            raise ValueError(
                "O conteúdo do JSON não é uma lista."
            )

        total_chunks = len(
            data
        )

        if total_chunks == 0:

            raise ValueError(
                "O arquivo não possui chunks."
            )

        valid_chunks = 0

        missing_text = 0
        missing_summary = 0
        missing_cleaned_summary = 0

        for record in data:

            if not isinstance(
                record,
                dict
            ):

                continue

            if record.get(
                "text"
            ):

                valid_chunks += 1

            else:

                missing_text += 1

            if not record.get(
                "summary"
            ):

                missing_summary += 1

            if not record.get(
                "cleaned_summary"
            ):

                missing_cleaned_summary += 1

        article_name = (
            file.name
            .split(
                "_chunksrecord_"
            )[0]
        )

        article_parts = (
            article_name
            .split(
                "-",
                2
            )
        )

        author = (
            article_parts[0]
            if len(article_parts) > 0
            else ""
        )

        year = (
            article_parts[1]
            if len(article_parts) > 1
            else ""
        )

        title = (
            article_parts[2]
            if len(article_parts) > 2
            else article_name
        )

        manifest_records.append(
            {
                "article_name": article_name,
                "author": author,
                "year": year,
                "title": title,
                "json_file": file.name,
                "total_chunks": total_chunks,
                "chunks_with_text": valid_chunks,
                "missing_text": missing_text,
                "missing_summary": missing_summary,
                "missing_cleaned_summary": missing_cleaned_summary,
                "status": "OK"
            }
        )

    except Exception as error:

        invalid_files.append(
            {
                "file": file.name,
                "error": str(error)
            }
        )

print_info(
    "Arquivos analisados:",
    len(json_files)
)

print_info(
    "Arquivos válidos:",
    len(manifest_records)
)

print_info(
    "Arquivos inválidos:",
    len(invalid_files)
)

if invalid_files:

    print_warning(
        "Existem arquivos JSON com problemas."
    )

else:

    print_success(
        "Todos os arquivos JSON foram validados."
    )

print("=" * 70)


 AUDITORIA DOS ARQUIVOS JSON
Arquivos analisados:      648
Arquivos válidos:         648
Arquivos inválidos:       0

✅ Todos os arquivos JSON foram validados.


In [ ]:
# Cria a tabela mestre da base

manifest_df = pd.DataFrame(
    manifest_records
)

manifest_df = manifest_df.sort_values(
    by=[
        "author",
        "year",
        "title"
    ]
).reset_index(
    drop=True
)

manifest_df.insert(
    0,
    "article_id",
    range(
        1,
        len(manifest_df) + 1
    )
)

print_header(
    "TABELA MESTRE DA BASE"
)

print_info(
    "Artigos:",
    len(manifest_df)
)

print_info(
    "Total de chunks:",
    manifest_df[
        "total_chunks"
    ].sum()
)

print_success(
    "Tabela mestre criada com sucesso."
)

print("=" * 70)


 TABELA MESTRE DA BASE
Artigos:                  648
Total de chunks:          6844

✅ Tabela mestre criada com sucesso.


In [ ]:
# Exibe uma amostra da tabela mestre para conferência

print_header(
    "AMOSTRA DA TABELA MESTRE"
)

display(
    manifest_df.head(
        10
    )
)

print_success(
    "Amostra da tabela mestre exibida com sucesso."
)

print("=" * 70)


 AMOSTRA DA TABELA MESTRE


,article_id,article_name,author,year,title,json_file,total_chunks,chunks_with_text,missing_text,missing_summary,missing_cleaned_summary,status
0,1,Abbasi-2017-Technology roadmap for the Creativ,Abbasi,2017,Technology roadmap for the Creativ,Abbasi-2017-Technology roadmap for the Creativ...,12,12,0,0,0,OK
1,2,Abe-2006-2nd generation business modeling_ Sma,Abe,2006,2nd generation business modeling_ Sma,Abe-2006-2nd generation business modeling_ Sma...,5,5,0,0,0,OK
2,3,Abe-2007-Integration studies of business model,Abe,2007,Integration studies of business model,Abe-2007-Integration studies of business model...,5,5,0,0,0,OK
3,4,Abe-2008-Towards systematic innovation methods,Abe,2008,Towards systematic innovation methods,Abe-2008-Towards systematic innovation methods...,6,6,0,0,0,OK
4,5,Abe-2009-A challenge for service concept model,Abe,2009,A challenge for service concept model,Abe-2009-A challenge for service concept model...,7,7,0,0,0,OK
5,6,Abe-2009-Integrating business modeling and roa,Abe,2009,Integrating business modeling and roa,Abe-2009-Integrating business modeling and roa...,7,7,0,0,0,OK
6,7,Abe-2010-Empowering technology marketing by th,Abe,2010,Empowering technology marketing by th,Abe-2010-Empowering technology marketing by th...,8,8,0,0,0,OK
7,8,Abuseem-2020-Technology mapping_ Definitions,Abuseem,2020,Technology mapping_ Definitions,Abuseem-2020-Technology mapping_ Definitions_c...,7,7,0,0,0,OK
8,9,Afanasiev-2010-Use of technology roadmapping i,Afanasiev,2010,Use of technology roadmapping i,Afanasiev-2010-Use of technology roadmapping i...,3,3,0,0,0,OK
9,10,Ahlqvist-2007-Nordic ICT foresight_ Futures of,Ahlqvist,2007,Nordic ICT foresight_ Futures of,Ahlqvist-2007-Nordic ICT foresight_ Futures of...,65,65,0,0,0,OK



✅ Amostra da tabela mestre exibida com sucesso.


In [ ]:
# Salva a tabela mestre no Google Drive em CSV e Excel

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_EXCEL_FILE = (
    RESULTS_DIR
    / "manifesto_base.xlsx"
)

manifest_df.to_csv(
    MANIFEST_FILE,
    index=False,
    encoding="utf-8-sig"
)

manifest_df.to_excel(
    MANIFEST_EXCEL_FILE,
    index=False
)

print_header(
    "SALVAMENTO DO MANIFESTO"
)

print_info(
    "Arquivo CSV:",
    MANIFEST_FILE
)

print_info(
    "Arquivo Excel:",
    MANIFEST_EXCEL_FILE
)

print_info(
    "Artigos:",
    len(manifest_df)
)

print_info(
    "Chunks:",
    manifest_df[
        "total_chunks"
    ].sum()
)

print_success(
    "Manifesto salvo em CSV e Excel com sucesso."
)

print("=" * 70)


 SALVAMENTO DO MANIFESTO
Arquivo CSV:              /content/drive/MyDrive/RAG_Novo_Embeddings/04_Resultados/manifesto_base.csv
Arquivo Excel:            /content/drive/MyDrive/RAG_Novo_Embeddings/04_Resultados/manifesto_base.xlsx
Artigos:                  648
Chunks:                   6844

✅ Manifesto salvo em CSV e Excel com sucesso.


In [ ]:
# Exibe eventuais arquivos inválidos

print_header(
    "RELATÓRIO DE PROBLEMAS"
)

if not invalid_files:

    print_success(
        "Nenhum arquivo JSON inválido foi encontrado."
    )

else:

    for item in invalid_files:

        print_error(
            item["file"]
        )

        print_info(
            "Erro:",
            item["error"]
        )

print("=" * 70)


 RELATÓRIO DE PROBLEMAS

✅ Nenhum arquivo JSON inválido foi encontrado.


Verificação Final do Notebook

In [ ]:
# Realiza a verificação final da auditoria

print_header(
    "VERIFICAÇÃO FINAL"
)

print_info(
    "JSONs encontrados:",
    len(json_files)
)

print_info(
    "Artigos válidos:",
    len(manifest_df)
)

print_info(
    "Total de chunks:",
    manifest_df[
        "total_chunks"
    ].sum()
)

print_info(
    "Média de chunks/artigo:",
    round(
        manifest_df[
            "total_chunks"
        ].mean(),
        2
    )
)

print_info(
    "Arquivos inválidos:",
    len(invalid_files)
)

print_info(
    "Manifesto CSV:",
    MANIFEST_FILE.name
)

print_info(
    "Manifesto Excel:",
    MANIFEST_EXCEL_FILE.name
)

if len(invalid_files) == 0:

    print_success(
        "Notebook 01 concluído com sucesso."
    )

else:

    print_warning(
        "Notebook concluído, mas existem arquivos que precisam ser revisados."
    )

print("=" * 70)


 VERIFICAÇÃO FINAL
JSONs encontrados:        648
Artigos válidos:          648
Total de chunks:          6844
Média de chunks/artigo:   10.56
Arquivos inválidos:       0
Manifesto CSV:            manifesto_base.csv
Manifesto Excel:          manifesto_base.xlsx

✅ Notebook 01 concluído com sucesso.


Estatísticas da Base

In [ ]:
# Exibe estatísticas adicionais sobre a distribuição de chunks por artigo

print_header(
    "ESTATÍSTICAS DA BASE"
)

largest_article = manifest_df.loc[
    manifest_df["total_chunks"].idxmax()
]

smallest_article = manifest_df.loc[
    manifest_df["total_chunks"].idxmin()
]

print_info(
    "Maior artigo:",
    largest_article["article_name"]
)

print_info(
    "Chunks do maior artigo:",
    int(
        largest_article["total_chunks"]
    )
)

print()

print_info(
    "Menor artigo:",
    smallest_article["article_name"]
)

print_info(
    "Chunks do menor artigo:",
    int(
        smallest_article["total_chunks"]
    )
)

print()

print_info(
    "Média de chunks/artigo:",
    round(
        manifest_df["total_chunks"].mean(),
        2
    )
)

print_info(
    "Mediana de chunks/artigo:",
    round(
        manifest_df["total_chunks"].median(),
        2
    )
)

print_success(
    "Estatísticas da base calculadas com sucesso."
)

print("=" * 70)


 ESTATÍSTICAS DA BASE
Maior artigo:             Mirzaei-2016-The Framework for Future Business
Chunks do maior artigo:   279

Menor artigo:             Anderson-2008-Air Force Materiel Command Roadm
Chunks do menor artigo:   1

Média de chunks/artigo:   10.56
Mediana de chunks/artigo: 9.0

✅ Estatísticas da base calculadas com sucesso.
